In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/sahilnagpal/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding = OpenAIEmbeddings(model="text-embedding-ada-002")

vector_store = Chroma(persist_directory="/Users/sahilnagpal/Desktop/AI-Square/RAG/intro-data-science-vectorstore/", embedding_function=embedding)

/var/folders/4t/t35871352tzf_hvp72_bzkcw0000gp/T/ipykernel_94142/2024856626.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding = OpenAIEmbeddings(model="text-embedding-ada-002")
/var/folders/4t/t35871352tzf_hvp72_bzkcw0000gp/T/ipykernel_94142/2024856626.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(persist_directory="/Users/sahilnagpal/Desktop/AI-Squa

In [4]:
len(vector_store.get()['documents'])

20

In [5]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 2, "lambda_mult": 0.7})

In [6]:
TEMPLATE = """
Answer the following questions:
{question}

To answer the question, use the following pieces of context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources:*Lecture Title*
where *Lecture Title* is the name of the lecture.
"""

In [7]:
prompt_template = PromptTemplate.from_template(TEMPLATE)

In [8]:
chat = ChatOpenAI(
    model="gpt-4",
    temperature=0,
    max_tokens=500,
    streaming=True,
    seed=365
)

In [9]:
question = "What software does data scientists use ?"

In [ ]:
chain = (
    {
    'question' : RunnablePassthrough(),
    'context' : retriever}
    | prompt_template
    | chat
    | StrOutputParser()
)

print(chain.invoke(question))